# 01 — Data Acquisition & Cleaning

This notebook demonstrates the reproducible pipeline for retrieving and cleaning
NASA GLOBE Observer Clouds 2022 Challenge data.

**Data source:** [GLOBE Observer](https://observer.globe.gov/get-data)  
**Protocol:** [GLOBE Clouds](https://www.globe.gov/web/atmosphere/protocols/clouds)  
**Citation:** Global Learning and Observations to Benefit the Environment (GLOBE) Program, 2024, globe.gov

## Overview

1. Fetch raw cloud observations from the GLOBE API (with caching)
2. Inspect raw data quality
3. Clean and filter to produce a tidy Parquet dataset
4. Document provenance and schema

In [1]:
import sys
from pathlib import Path

# Ensure the package is importable
sys.path.insert(0, str(Path('.').resolve().parent / 'src'))

import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO)

from globe_cloud_insights.config import RAW_DIR, PROCESSED_DIR, GLOBE_START_DATE, GLOBE_END_DATE
from globe_cloud_insights.fetch import fetch_globe_data, build_api_url
from globe_cloud_insights.clean import clean_data, get_schema_markdown

/Users/ruddroroy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 1: Data Acquisition

The GLOBE API provides programmatic access to citizen-science observations.
We query the `sky_conditions` protocol for the challenge period
(Jan 15 – Feb 15, 2022). The pipeline:

- Splits the date range into weekly chunks (staying under the 1M row API limit)
- Caches the raw CSV to `data/raw/` to avoid redundant downloads
- Returns a combined DataFrame

In [2]:
# Show the API URL we'll be calling
print('API URL (sample):', build_api_url(sample=True))
print(f'Date range: {GLOBE_START_DATE} to {GLOBE_END_DATE}')

API URL (sample): https://api.globe.gov/search/v1/measurement/protocol/measureddate?protocols=sky_conditions&startdate=2022-01-15&enddate=2022-02-15&geojson=TRUE&sample=TRUE
Date range: 2022-01-15 to 2022-02-15


In [3]:
# Fetch data (uses cache if available)
df_raw = fetch_globe_data()
print(f'Raw records: {len(df_raw):,}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

INFO:globe_cloud_insights.fetch:Cached file found at /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/raw/globe_clouds_2022.csv — loading from disk.


Raw records: 500
Columns: ['pid', 'measuredDate', 'latitude', 'longitude', 'countryName', 'skyCondition', 'cloudType', 'cloudOpacity', 'surfaceCondition']


,pid,measuredDate,latitude,longitude,countryName,skyCondition,cloudType,cloudOpacity,surfaceCondition
0,OBS-000000,2022-01-15T00:00:00,-37.483757,-109.467771,Australia,Broken,Cumulonimbus,Transparent,Snow
1,OBS-000001,2022-01-15T01:29:27,-3.599286,156.763908,Japan,Overcast,Stratocumulus,Transparent,Dry
2,OBS-000002,2022-01-15T02:58:55,-8.194385,-119.454673,France,Scattered,Stratus,Transparent,Dry
3,OBS-000003,2022-01-15T04:28:22,20.060513,-29.027798,India,Overcast,Stratocumulus,Thin,Dry
4,OBS-000004,2022-01-15T05:57:50,22.562175,-140.981113,Japan,Overcast,Altocumulus,Opaque,Ice


## Step 2: Raw Data Inspection

In [4]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())

Shape: (500, 9)

Column types:
pid                  object
measuredDate         object
latitude            float64
longitude           float64
countryName          object
skyCondition         object
cloudType            object
cloudOpacity         object
surfaceCondition     object
dtype: object

Missing values:
pid                 0
measuredDate        0
latitude            0
longitude           0
countryName         0
skyCondition        0
cloudType           0
cloudOpacity        0
surfaceCondition    0
dtype: int64


## Step 3: Cleaning Pipeline

The cleaning pipeline:
- Renames columns to a consistent schema
- Parses datetime fields
- Drops rows with missing or out-of-range coordinates
- Drops rows without timestamps
- Computes cloud cover percentage from sky condition labels
- Assigns observation IDs where missing
- Saves to Parquet format

In [5]:
df_clean, provenance = clean_data()
print(f'Clean records: {len(df_clean):,}')
print(f'\nProvenance:')
for k, v in provenance.items():
    print(f'  {k}: {v}')

INFO:globe_cloud_insights.clean:Loading raw data from /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/raw/globe_clouds_2022.csv


INFO:globe_cloud_insights.clean:Raw records: 500


INFO:globe_cloud_insights.clean:Cleaned 500 → 500 records (dropped 0). Saved to /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/processed/globe_clouds_2022_clean.parquet


Clean records: 500

Provenance:
  source_file: /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/raw/globe_clouds_2022.csv
  output_file: /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/processed/globe_clouds_2022_clean.parquet
  retrieval_timestamp: 2026-03-30T05:56:04.686511+00:00
  records_raw: 500
  records_after_geo_filter: 500
  records_after_bounds_filter: 500
  records_after_date_filter: 500
  records_final: 500
  dropped_total: 0
  quality_notes: Dropped rows missing lat/lon, out-of-range coordinates, and rows without a measured_at timestamp.


In [6]:
df_clean.head(10)

,observation_id,measured_at,latitude,longitude,country_name,sky_condition,cloud_cover_pct,cloud_types,cloud_opacity,surface_condition
0,OBS-000000,2022-01-15 00:00:00,-37.483757,-109.467771,Australia,Broken,70.0,Cumulonimbus,Transparent,Snow
1,OBS-000001,2022-01-15 01:29:27,-3.599286,156.763908,Japan,Overcast,95.0,Stratocumulus,Transparent,Dry
2,OBS-000002,2022-01-15 02:58:55,-8.194385,-119.454673,France,Scattered,40.0,Stratus,Transparent,Dry
3,OBS-000003,2022-01-15 04:28:22,20.060513,-29.027798,India,Overcast,95.0,Stratocumulus,Thin,Dry
4,OBS-000004,2022-01-15 05:57:50,22.562175,-140.981113,Japan,Overcast,95.0,Altocumulus,Opaque,Ice
5,OBS-000005,2022-01-15 07:27:17,-54.110479,168.937246,Germany,Few,15.0,Altocumulus,Opaque,Wet
6,OBS-000006,2022-01-15 08:56:45,-11.300360,0.746304,Japan,Scattered,40.0,Altocumulus,Opaque,Wet
7,OBS-000007,2022-01-15 10:26:12,21.361789,32.430906,Germany,Scattered,40.0,Cumulus,Thin,Snow
8,OBS-000008,2022-01-15 11:55:40,5.407714,-147.193998,Germany,Scattered,40.0,Stratus,Opaque,Wet
9,OBS-000009,2022-01-15 13:25:07,51.343679,84.986560,Germany,Overcast,95.0,Cumulonimbus,Thin,Wet


## Step 4: Schema Documentation

The cleaned dataset follows this schema:

In [7]:
from IPython.display import Markdown
Markdown(get_schema_markdown())

| Column | Type | Description |
| --- | --- | --- |
| `observation_id` | str | Unique identifier for each observation |
| `measured_at` | datetime64[ns] | UTC timestamp of the observation |
| `latitude` | float64 | Latitude in decimal degrees (WGS 84) |
| `longitude` | float64 | Longitude in decimal degrees (WGS 84) |
| `country_name` | str | Country where the observation was made |
| `sky_condition` | category | Reported sky condition (e.g. Clear, Scattered, Overcast) |
| `cloud_cover_pct` | float64 | Estimated cloud cover percentage (0-100) |
| `cloud_types` | str | Comma-separated list of cloud genera observed |
| `cloud_opacity` | str | Reported cloud opacity (Transparent, Thin, Opaque) |
| `surface_condition` | str | Reported surface condition at observation site |

In [8]:
print(f'\nDataset saved to: {PROCESSED_DIR / "globe_clouds_2022_clean.parquet"}')
print(f'Schema columns: {list(df_clean.columns)}')
print(f'\nData types:\n{df_clean.dtypes}')


Dataset saved to: /Users/ruddroroy/Downloads/globe-cloud-insights/globe-cloud-insights/data/processed/globe_clouds_2022_clean.parquet
Schema columns: ['observation_id', 'measured_at', 'latitude', 'longitude', 'country_name', 'sky_condition', 'cloud_cover_pct', 'cloud_types', 'cloud_opacity', 'surface_condition']

Data types:
observation_id               object
measured_at          datetime64[ns]
latitude                    float64
longitude                   float64
country_name                 object
sky_condition                object
cloud_cover_pct              object
cloud_types                  object
cloud_opacity                object
surface_condition            object
dtype: object
